# 04 — Patient Document Availability and Linkage Readiness Audit

**Status:** `PENDING MANUAL EXECUTION`

**Status:** `PENDING` (Prepared for user execution)

## Objective
To assess the availability, completeness, organization, and linkage readiness of supporting documents associated with the synthetic patient records in the current OFPHIL V2 dataset (`data/raw/tabular/samples/charity-patient.csv`).

### Key Distinctions
1. **Document indicators recorded in the patient CSV** (e.g., `Present`, `Absent`, `UNRECORDED`, `NaN`)
2. **Actual supporting-document files** in `data/raw/supporting document/supporting_docu/`
3. **Verified patient-document relationships** (requires a reliable patient identifier + document manifest)

### Relationship with ResNet-50 and PaddleOCR
- **ResNet-50**: Visual processing, feature extraction, and authentic vs. tampered document screening on actual document images.
- **PaddleOCR**: Text and structured field extraction from actual document images.
- **Patient Document Linkage**: Associates verified document records with the correct patient before document-derived features enter eligibility classification (Category 6: *Eligibility Documentation and Certification*).
- **Important**: ResNet-50 and PaddleOCR cannot perform meaningful document-image processing on synthetic tabular records that lack corresponding document images.


In [13]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "tabular" / "samples" / "charity-patient.csv"
DOC_REPO = PROJECT_ROOT / "data" / "raw" / "supporting document" / "supporting_docu"
OUT_DIR = PROJECT_ROOT / "data" / "interim" / "linkage_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_CSV)
print(f"Loaded {len(df)} records x {df.shape[1]} columns from {RAW_CSV.relative_to(PROJECT_ROOT)}")


Loaded 1000 records x 71 columns from data\raw\tabular\samples\charity-patient.csv


## 1. Patient Dataset Document Indicator Audit
Inspects all document-related columns actually present in `charity-patient.csv`, distinguishing `Present`, `Absent`, `UNRECORDED`, and `NaN` (null) values.


In [14]:
DOC_INDICATOR_COLS = [
    "medical_abstract_or_certificate",
    "medical_record_diagnosis_treatment_plan",
    "proof_of_income_employment",
    "certificate_of_indigency",
    "social_welfare_assessment_certification",
    "government_assistance_certification",
    "ngo_charity_assistance_certification",
    "other_eligibility_certifications",
    "government_id",
    "payslip_document",
    "utility_bill_document",
    "referral_letter",
    "philhealth_document",
    "total_supporting_documents",
    "document_category_diversity",
]

summary_rows = []
for col in DOC_INDICATOR_COLS:
    if col not in df.columns:
        summary_rows.append({"column": col, "exists": False})
        continue
    s = df[col]
    null_cnt = int(s.isna().sum())
    str_vals = s.dropna().astype(str).str.strip().str.upper()
    summary_rows.append({
        "column": col,
        "exists": True,
        "total_records": len(df),
        "non_null": int(s.notna().sum()),
        "missing_null": null_cnt,
        "present_count": int((str_vals == "PRESENT").sum()),
        "absent_count": int((str_vals == "ABSENT").sum()),
        "unrecorded_count": int((str_vals == "UNRECORDED").sum()),
    })

indicator_df = pd.DataFrame(summary_rows)
indicator_df.to_csv(OUT_DIR / "document_indicator_summary.csv", index=False)
display(indicator_df)


,column,exists,total_records,non_null,missing_null,present_count,absent_count,unrecorded_count
0,medical_abstract_or_certificate,True,1000,786,214,476,310,0
1,medical_record_diagnosis_treatment_plan,True,1000,786,214,476,310,0
2,proof_of_income_employment,True,1000,784,216,108,676,0
3,certificate_of_indigency,True,1000,822,178,472,350,0
4,social_welfare_assessment_certification,True,1000,571,429,99,472,0
5,government_assistance_certification,True,1000,644,356,38,606,0
6,ngo_charity_assistance_certification,True,1000,571,429,37,534,0
7,other_eligibility_certifications,True,1000,183,817,0,4,0
8,government_id,True,1000,753,247,393,360,0
9,payslip_document,True,1000,690,310,87,603,0


## 2. Supporting-Document Repository Inventory
Scans `data/raw/supporting document/supporting_docu/` for actual files, extensions, directory structure, and duplicate hashes. Files without verifiable patient ownership are classified as `UNLINKED / UNVERIFIED`.


In [15]:
repo_rows = []
if DOC_REPO.exists():
    for p in sorted(DOC_REPO.rglob("*")):
        if p.is_file():
            h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
            rel = p.relative_to(DOC_REPO)
            repo_rows.append({
                "relative_path": str(rel).replace("\\", "/"),
                "category": rel.parts[1] if len(rel.parts) > 2 else (rel.parts[0] if len(rel.parts) > 1 else "root"),
                "extension": p.suffix.lower(),
                "size_bytes": p.stat().st_size,
                "sha256_prefix": h,
                "linkage_status": "UNLINKED / UNVERIFIED",
            })

repo_df = pd.DataFrame(repo_rows)
repo_df.to_csv(OUT_DIR / "supporting_document_inventory.csv", index=False)
print(f"Total files in repository: {len(repo_df)}")
if not repo_df.empty:
    print("By extension:", repo_df["extension"].value_counts().to_dict())
    print("Duplicate hashes:", int(repo_df["sha256_prefix"].duplicated().sum()))


Total files in repository: 226
By extension: {'.jpg': 181, '.jpeg': 45}
Duplicate hashes: 1


## 3. Patient-Document Linkage Readiness Assessment
Evaluates whether the current dataset and repository satisfy technical requirements for verified patient-document linkage.


In [16]:
has_patient_id = any(c in df.columns for c in ["patient_id", "patient_code", "id"])
readiness = {
    "csv_document_indicator_analysis": "READY",
    "supporting_document_inventory": "READY" if DOC_REPO.exists() else "NOT READY",
    "reliable_patient_identifier_in_csv": "READY" if has_patient_id else "NOT READY",
    "patient_to_document_mapping_table": "NOT AVAILABLE",
    "actual_patient_document_matching": "NOT READY",
    "resnet50_paddleocr_linked_execution": "NOT APPLICABLE (no verified patient-to-image links for synthetic 1k CSV)",
}
(OUT_DIR / "linkage_readiness_summary.json").write_text(json.dumps(readiness, indent=2), encoding="utf-8")
print(json.dumps(readiness, indent=2))


{
  "csv_document_indicator_analysis": "READY",
  "supporting_document_inventory": "READY",
  "reliable_patient_identifier_in_csv": "NOT READY",
  "patient_to_document_mapping_table": "NOT AVAILABLE",
  "actual_patient_document_matching": "NOT READY",
  "resnet50_paddleocr_linked_execution": "NOT APPLICABLE (no verified patient-to-image links for synthetic 1k CSV)"
}
